# 1 — Text labels, multilingual patterns, and preprocessing

This notebook creates sentiment targets from **review text rather than star ratings**.
VADER and TextBlob provide high-confidence weak labels for real app reviews. A separate,
team-curated training file adds English, Malay, Manglish, emoji, negation, and contrastive
sentence patterns. Curated patterns are training-only and never enter the main test set.

The post-contrast clause after *but/tapi/however/yet* receives explicit focus features.
Label text and model text are prepared separately, so VADER/TextBlob never receive
machine-only tokens such as `NOT_good`. Repeated letters, known informal spellings,
and close variants of the project's sentiment vocabulary are normalised before the
linguistic phrase rules run a second time; therefore `not gud` becomes `bad`.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'dataset').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))
DATA_DIR = PROJECT_ROOT / 'dataset'
MODEL_DIR = PROJECT_ROOT / 'model'
print('Project root:', PROJECT_ROOT)

import pandas as pd
from sentiment_pipeline import normalize_for_labeler, focus_for_labeler, preprocess_for_model
from project_workflow import build_dataset

Project root: C:\Users\Fnjtnpcl\Documents\Codex\2026-08-16\referenced-chatgpt-conversation-this-is-an\work\nlp_v31\nlp_sentiment_project_final_fixed (1)\project


In [2]:
examples = [
    "Food delicious but I don't like it",
    "Delivery fast tapi food tak sedap",
    "Not bad",
    "Tak puas hati",
    "Not gud",
    "Food is not guudd",
    "Food 🤮",
]
pd.DataFrame({
    'raw_review': examples,
    'label_view': [focus_for_labeler(x) for x in examples],
    'model_features': [preprocess_for_model(x) for x in examples],
})

,raw_review,label_view,model_features
0,Food delicious but I don't like it,i dislike it,food delicious but i dislike it CONTRAST_FOCUS...
1,Delivery fast tapi food tak sedap,food bad tasting,delivery fast but food bad tasting CONTRAST_FO...
2,Not bad,acceptable good,acceptable good
3,Tak puas hati,dissatisfied,dissatisfied
4,Not gud,bad,bad
5,Food is not guudd,food is bad,food is bad
6,Food 🤮,food disgusting,food disgusting


In [3]:
summary = build_dataset(PROJECT_ROOT)
summary

{'candidate_rows': 59205,
 'real_balanced_rows': 21585,
 'curated_train_patterns': 630,
 'train_rows': 17897,
 'test_rows': 4317}

In [4]:
train = pd.read_csv(DATA_DIR / 'train_split.csv')
test = pd.read_csv(DATA_DIR / 'test_split.csv')
patterns = pd.read_csv(DATA_DIR / 'curated_training_patterns.csv')
assert set(train['normalized_key']).isdisjoint(set(test['normalized_key']))
print('Training label counts:')
print(train['text_sentiment'].value_counts().sort_index())
print()
print('Real held-out test counts:')
print(test['text_sentiment'].value_counts().sort_index())
print()
print('Curated training patterns:', len(patterns))
patterns[['review', 'text_sentiment', 'language_pattern', 'pattern_type']].sample(12, random_state=42)

Training label counts:
text_sentiment
negative    6016
neutral     5901
positive    5980
Name: count, dtype: int64

Real held-out test counts:
text_sentiment
negative    1439
neutral     1439
positive    1439
Name: count, dtype: int64

Curated training patterns: 630


,review,text_sentiment,language_pattern,pattern_type
497,Food was okay,neutral,English/Malay/Manglish/emoji,direct_or_negation
244,Grab review: Service memang teruk,negative,English/Malay/Manglish/emoji,direct_or_negation
552,Foodpanda order: Bayaran guna kad,neutral,English/Malay/Manglish/emoji,direct_or_negation
213,Food basi dan sejuk,negative,English/Malay/Manglish/emoji,direct_or_negation
549,Foodpanda order: Saya order dua makanan,neutral,English/Malay/Manglish/emoji,direct_or_negation
506,Delivery arrived at nine pm,neutral,English/Malay/Manglish/emoji,direct_or_negation
605,"Honestly, The driver called once",neutral,English/Malay/Manglish/emoji,direct_or_negation
77,"Driver sampai cepat, tetapi service teruk gila.",negative,English/Malay/Manglish,contrast_tail
531,Grab review: The receipt shows twelve ringgit,neutral,English/Malay/Manglish/emoji,direct_or_negation
145,"Harga agak mahal, but makanan sedap gila.",positive,English/Malay/Manglish,contrast_tail


Main accuracy is later calculated only on unseen, real app-review rows. The curated
patterns make important linguistic structures learnable but do not inflate the main
test score. The separate challenge CSV is a functional diagnostic, not a benchmark.